In [15]:
import re
import os
import tempfile
from pathlib import Path
from collections import defaultdict

import boto3
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
from shapely.geometry import mapping

In [16]:
def list_s3_keys(bucket, prefix=""):
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    rows = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            rows.append({
                "bucket": bucket,
                "key": obj["Key"],
                "size": obj["Size"],
            })

    return pd.DataFrame(rows)

In [17]:
def extract_tile(key):
    m = re.search(r"(p\d{3}r\d{3})", key)
    return m.group(1) if m else None

def extract_date_group(key):
    m = re.search(r"(d\d{16})", key)
    if m:
        return m.group(1)
    m = re.search(r"(?<!\d)(\d{8})(?!\d)", key)
    if m:
        return m.group(1)
    return None

def classify_file(key):
    kl = key.lower()
    if kl.endswith(".shp"):
        return "shapefile"
    if kl.endswith(".tif"):
        if "_dlj_" in kl:
            return "dlj"
        if "_dll_" in kl:
            return "dll"
        if "vi-ndvi" in kl:
            return "ndvi_variant"
        return "raster_other"
    return "other"

In [18]:
df_my = list_s3_keys("dcceew-eds-data", "AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/")
df_qld = list_s3_keys("dcceew-rs-data", "")  # adjust prefix if needed

for df in [df_my, df_qld]:
    df["tile"] = df["key"].apply(extract_tile)
    df["date_group"] = df["key"].apply(extract_date_group)
    df["file_type"] = df["key"].apply(classify_file)

In [19]:
def summarise_df(df, name):
    print(f"\n===== {name} =====")
    print("Total files:", len(df))
    print("\nFile types:")
    print(df["file_type"].value_counts(dropna=False))

    print("\nTiles:")
    print(df["tile"].value_counts().head(10))

    print("\nDate groups:")
    print(df["date_group"].value_counts().head(10))

In [20]:
summarise_df(df_my, "MY EDS")
summarise_df(df_qld, "QLD / RS DATA")


===== MY EDS =====
Total files: 11811

File types:
file_type
raster_other    11757
other              38
shapefile           8
dlj                 4
dll                 4
Name: count, dtype: int64

Tiles:
tile
p089r079    439
p089r080    413
p089r078    393
p091r085    380
p092r080    356
p090r077    354
p091r076    330
p089r081    320
p092r081    316
p091r084    308
Name: count, dtype: int64

Date groups:
date_group
20251003    42
20251112    40
20211211    40
20230928    38
20231022    38
20241219    38
20250917    38
20211203    38
20220112    38
20220205    38
Name: count, dtype: int64

===== QLD / RS DATA =====
Total files: 3071

File types:
file_type
other           1738
raster_other     967
shapefile        366
Name: count, dtype: int64

Tiles:
tile
p091r077    32
p092r076    26
p090r079    25
p090r086    24
p092r077    24
p094r077    23
p092r088    23
p099r069    23
p095r073    22
p091r076    22
Name: count, dtype: int64

Date groups:
date_group
d2025101320260218    63
d202511

In [21]:
df_my["key"].head(20)
df_qld["key"].head(20)

0       AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/s3_test.txt
1     ancil_files/EPBC_SPATIAL_REFERRALS/EPBC_SPATIA...
2     ancil_files/EPBC_SPATIAL_REFERRALS/EPBC_SPATIA...
3     ancil_files/EPBC_SPATIAL_REFERRALS/EPBC_SPATIA...
4     ancil_files/EPBC_SPATIAL_REFERRALS/EPBC_SPATIA...
5     ancil_files/EPBC_SPATIAL_REFERRALS/EPBC_SPATIA...
6     ancil_files/EPBC_SPATIAL_REFERRALS/EPBC_SPATIA...
7     ancil_files/EPBC_SPATIAL_REFERRALS/EPBC_SPATIA...
8     ancil_files/EPBC_SPATIAL_REFERRALS/EPBC_SPATIA...
9     ancil_files/epbc_referrals/EPBC_SPATIAL_REFERR...
10    ancil_files/epbc_referrals/EPBC_SPATIAL_REFERR...
11    ancil_files/epbc_referrals/EPBC_SPATIAL_REFERR...
12    ancil_files/epbc_referrals/EPBC_SPATIAL_REFERR...
13    ancil_files/epbc_referrals/EPBC_SPATIAL_REFERR...
14    ancil_files/epbc_referrals/EPBC_SPATIAL_REFERR...
15    ancil_files/epbc_referrals/EPBC_SPATIAL_REFERR...
16    ancil_files/epbc_referrals/EPBC_SPATIAL_REFERR...
17    ancil_files/epbc_referrals/EPBC_SPATIAL_RE

In [22]:
df_my[df_my["file_type"] == "shapefile"]["key"].head(20)
df_qld[df_qld["file_type"] == "shapefile"]["key"].head(20)

df_my[df_my["file_type"].str.contains("dlj|dll", na=False)]["key"].head(20)

822     AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
823     AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
1234    AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
1235    AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
1553    AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
1554    AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
1846    AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
1847    AROAZ6PFZYT4B4C7MNRHV:robotmcgregor/eds/tiles/...
Name: key, dtype: object

In [23]:
common_tiles = set(df_my["tile"].dropna()) & set(df_qld["tile"].dropna())
print("Common tiles:", len(common_tiles))
print(sorted(list(common_tiles))[:10])

Common tiles: 14
['p089r078', 'p089r084', 'p090r077', 'p090r079', 'p090r086', 'p090r088', 'p090r090', 'p091r076', 'p091r077', 'p091r087']


In [24]:
df_my_dates = df_my[df_my["date_group"].notna()]
df_qld_dates = df_qld[df_qld["date_group"].notna()]

common_dates = set(df_my_dates["date_group"]) & set(df_qld_dates["date_group"])

print("Common date groups:", len(common_dates))
print(list(common_dates)[:10])

Common date groups: 2
['20240929', '20240721']


In [25]:
qld_shapes = df_qld[df_qld["file_type"] == "shapefile"].copy()
my_rasters = df_my[df_my["file_type"].isin(["dlj", "dll", "ndvi_variant", "raster_other"])].copy()

matches = qld_shapes.merge(
    my_rasters,
    on=["tile", "date_group"],
    suffixes=("_qld", "_my")
)

matches[["tile", "date_group", "key_qld", "key_my"]].head()

,tile,date_group,key_qld,key_my


In [26]:
def download_s3_file(bucket, key, local_path):
    s3 = boto3.client("s3")
    local_path.parent.mkdir(parents=True, exist_ok=True)
    s3.download_file(bucket, key, str(local_path))

def download_shapefile_bundle(bucket, shp_key, out_dir):
    base = os.path.splitext(shp_key)[0]
    exts = [".shp", ".dbf", ".shx", ".prj", ".cpg"]
    local_files = []

    for ext in exts:
        key = base + ext
        local_path = Path(out_dir) / Path(key).name
        try:
            download_s3_file(bucket, key, local_path)
            local_files.append(local_path)
        except Exception:
            pass

    shp_local = Path(out_dir) / (Path(base).name + ".shp")
    return shp_local